# 02 — TF-IDF + LogReg baseline (Task B)

This is the reference number every transformer fine-tune must beat.

```bash
python -m src.baselines --task B
```

This notebook just reads the dumped metrics + predictions and produces the
diagnostic plots (confusion matrix, per-class F1, error inspection).

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

MODEL = Path('../models/baseline_B.joblib')
METRICS = Path('../results/baseline_B_metrics.json')
PREDS = Path('../results/baseline_B_predictions.parquet')

metrics = json.loads(METRICS.read_text())
preds = pd.read_parquet(PREDS)
metrics

In [ ]:
# confusion matrix on test split
test = preds[preds['split'] == 'test']
labels = ['negative', 'neutral', 'positive']
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test['y_true'], test['y_pred'], labels=labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'Test confusion (macro-F1={metrics["test"]["macro_f1"]:.3f})')

In [ ]:
# top informative features per class
pipe = joblib.load(MODEL)
vec = pipe.named_steps['tfidf']
clf = pipe.named_steps['clf']
vocab = np.array(vec.get_feature_names_out())
for i, cls in enumerate(clf.classes_):
    coefs = clf.coef_[i]
    top = np.argsort(coefs)[-20:][::-1]
    print(f'{cls}:', ', '.join(vocab[top].tolist()))

In [ ]:
# error inspection — most confident wrong predictions
df = pd.read_parquet('../data/processed/clean.parquet').set_index('review_id')
err = test[test['y_true'] != test['y_pred']].head(20)
for _, row in err.iterrows():
    review = df.loc[row['review_id'], 'review']
    print(f"true={row['y_true']:>8}  pred={row['y_pred']:>8}  | {review[:200]}…\n")